In [1]:
from pathlib import Path
import pandas as pd

# 1. Đọc dữ liệu Evaluation đã kiểm tra đủ 50 run
PROJECT_DIR = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / "results/evaluation/execution_accuracy_all_runs.csv").is_file()),
    None
)
assert PROJECT_DIR is not None, "Không tìm thấy thư mục dự án"

EVAL_DIR = PROJECT_DIR / "results/evaluation"
OUT_DIR = PROJECT_DIR / "results/extended_analysis/error_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ea = pd.read_csv(EVAL_DIR / "execution_accuracy_all_runs.csv", keep_default_na=False)
required = {"Run", "ID", "Method", "Status", "Correct"}
assert required.issubset(ea.columns), f"Thiếu cột: {required - set(ea.columns)}"

ea["Run"] = pd.to_numeric(ea["Run"], errors="raise").astype(int)
ea["ID"] = pd.to_numeric(ea["ID"], errors="raise").astype(int)
ea["Correct"] = pd.to_numeric(ea["Correct"], errors="raise").astype(int)
ea["Method"] = ea["Method"].astype(str).str.strip().str.upper()
ea["Status"] = ea["Status"].astype(str).str.strip().str.upper()

assert len(ea) == 50 * 43 * 2
assert set(ea["Run"]) == set(range(1, 51))
assert set(ea["ID"]) == set(range(1, 44))
assert set(ea["Method"]) == {"SQL", "SPARQL"}
assert ea["Correct"].isin([0, 1]).all()
assert not ea.duplicated(["Run", "ID", "Method"]).any()
assert ea.groupby(["Run", "Method"]).size().eq(43).all()
assert not ((ea["Status"] != "OK") & (ea["Correct"] == 1)).any()

# 2. Gắn Quadrant đã được đối chiếu với benchmark ở Notebook 05
quadrants = pd.read_csv(EVAL_DIR / "oea_by_quadrant.csv")
quadrants = quadrants[["ID", "Quadrant"]].drop_duplicates()
assert len(quadrants) == 43 and quadrants["ID"].is_unique
assert set(quadrants["Quadrant"]) == {"LQLS", "HQLS", "LQHS", "HQHS"}

ea = ea.merge(quadrants, on="ID", how="left", validate="many_to_one")
assert len(ea) == 4300 and ea["Quadrant"].notna().all()

print("=== ERROR ANALYSIS — 50 RUNS ===")
print("Runs:", ea["Run"].nunique())
print("Questions:", ea["ID"].nunique())
print("Evaluations:", len(ea))
print("Quadrants:", quadrants["Quadrant"].value_counts().to_dict())
print("Validation: OK")
display(ea.head())

=== ERROR ANALYSIS — 50 RUNS ===
Runs: 50
Questions: 43
Evaluations: 4300
Quadrants: {'LQLS': 12, 'HQLS': 11, 'HQHS': 10, 'LQHS': 10}
Validation: OK


,Run,ID,Method,Status,Correct,Quadrant
0,1,1,SQL,OK,0,HQHS
1,1,2,SQL,OK,1,LQLS
2,1,3,SQL,OK,0,LQHS
3,1,4,SQL,OK,0,HQHS
4,1,5,SQL,OK,0,LQHS


In [2]:
# 1. Ghép SQL/SPARQL trên cùng câu hỏi và cùng run
paired = ea.pivot(
    index=["Run", "ID", "Quadrant"],
    columns="Method",
    values="Correct"
).reset_index()
paired.columns.name = None

assert len(paired) == 50 * 43
assert paired[["SQL", "SPARQL"]].notna().all().all()

# 2. Bốn trường hợp so sánh cặp

def classify_pair(row):
    sql, sparql = int(row["SQL"]), int(row["SPARQL"])
    if sql == 1 and sparql == 1:
        return "Both Correct"
    if sql == 1:
        return "SQL Only"
    if sparql == 1:
        return "SPARQL Only"
    return "Both Wrong"

paired["Case"] = paired.apply(classify_pair, axis=1)
case_order = ["Both Correct", "SQL Only", "SPARQL Only", "Both Wrong"]

case_counts = paired["Case"].value_counts().reindex(case_order, fill_value=0)
assert int(case_counts.sum()) == 2150
assert int(case_counts["Both Correct"] + case_counts["SQL Only"]) == int(
    ea.loc[ea["Method"] == "SQL", "Correct"].sum()
)
assert int(case_counts["Both Correct"] + case_counts["SPARQL Only"]) == int(
    ea.loc[ea["Method"] == "SPARQL", "Correct"].sum()
)

print("=== PAIRED ERROR ANALYSIS ===")
display(case_counts.to_frame("Count"))

# 3. Thống kê theo Quadrant
quadrant_order = ["LQLS", "HQLS", "LQHS", "HQHS"]
quadrant_summary = pd.crosstab(
    paired["Quadrant"], paired["Case"]
).reindex(index=quadrant_order, columns=case_order, fill_value=0)
assert int(quadrant_summary.to_numpy().sum()) == 2150

print("\n=== BY QUADRANT ===")
display(quadrant_summary)

# 4. Xuất giữ nguyên tên file gốc
paired.to_csv(OUT_DIR / "paired_error_analysis.csv", index=False, encoding="utf-8-sig")
quadrant_summary.to_csv(OUT_DIR / "paired_by_quadrant.csv", encoding="utf-8-sig")
print("Saved: paired_error_analysis.csv, paired_by_quadrant.csv")

=== PAIRED ERROR ANALYSIS ===


,Count
Case,
Both Correct,759
SQL Only,224
SPARQL Only,650
Both Wrong,517



=== BY QUADRANT ===


Case,Both Correct,SQL Only,SPARQL Only,Both Wrong
Quadrant,,,,
LQLS,380,16,104,100
HQLS,222,68,146,114
LQHS,49,0,287,164
HQHS,108,140,113,139


Saved: paired_error_analysis.csv, paired_by_quadrant.csv


In [3]:
# 1. Phân biệt đúng, sai kết quả và không thực thi thành công
ea["Outcome"] = "Wrong Result"
ea.loc[ea["Correct"] == 1, "Outcome"] = "Correct"
ea.loc[ea["Status"] != "OK", "Outcome"] = "Execution Failure"

outcome_order = ["Correct", "Wrong Result", "Execution Failure"]

# 2. Thống kê tổng thể 2.150 lượt cho mỗi phương pháp
overall = pd.crosstab(ea["Method"], ea["Outcome"]).reindex(
    index=["SQL", "SPARQL"], columns=outcome_order, fill_value=0
)
assert overall.sum(axis=1).eq(2150).all()
assert int(overall["Correct"].sum()) == int(ea["Correct"].sum())

print("=== ERROR DISTRIBUTION ===")
display(overall)

# 3. Thống kê theo Quadrant
by_quadrant = (
    ea.groupby(["Quadrant", "Method", "Outcome"])
    .size().unstack(fill_value=0)
    .reindex(columns=outcome_order, fill_value=0)
)
assert int(by_quadrant.to_numpy().sum()) == 4300

print("\n=== ERROR DISTRIBUTION BY QUADRANT ===")
display(by_quadrant)

# 4. Xuất file dữ liệu phân tích lỗi
ea.to_csv(OUT_DIR / "error_details.csv", index=False, encoding="utf-8-sig")
overall.to_csv(OUT_DIR / "error_distribution.csv", encoding="utf-8-sig")
by_quadrant.to_csv(OUT_DIR / "error_by_quadrant.csv", encoding="utf-8-sig")
print("Saved: error_details.csv, error_distribution.csv, error_by_quadrant.csv")

=== ERROR DISTRIBUTION ===


Outcome,Correct,Wrong Result,Execution Failure
Method,,,
SQL,983,1086,81
SPARQL,1409,459,282



=== ERROR DISTRIBUTION BY QUADRANT ===


Outcome          Correct  Wrong Result  Execution Failure
Quadrant Method                                          
HQHS     SPARQL      221            74                205
         SQL         248           245                  7
HQLS     SPARQL      368           106                 76
         SQL         290           210                 50
LQHS     SPARQL      336           164                  0
         SQL          49           429                 22
LQLS     SPARQL      484           115                  1
         SQL         396           202                  2

Saved: error_details.csv, error_distribution.csv, error_by_quadrant.csv


In [4]:
# 1. Đọc log thực thi đã lưu của 50 run cho cả hai phương pháp
logs = []
for method in ["SQL", "SPARQL"]:
    for run_id in range(1, 51):
        log_file = (
            EVAL_DIR / f"{method.lower()}_runs"
            / f"run_{run_id:02d}" / "execution_log.csv"
        )
        assert log_file.is_file() and log_file.stat().st_size > 0, (
            f"Checkpoint thiếu hoặc rỗng: {log_file}"
        )
        log = pd.read_csv(log_file, keep_default_na=False)
        assert {"ID", "Status", "Error"}.issubset(log.columns), log_file
        assert len(log) == 43 and not log["ID"].duplicated().any(), log_file

        log["ID"] = pd.to_numeric(log["ID"], errors="raise").astype(int)
        assert set(log["ID"]) == set(range(1, 44)), log_file
        log["Run"] = run_id
        log["Method"] = method
        log["Status"] = log["Status"].astype(str).str.strip().str.upper()
        logs.append(log[["Run", "ID", "Method", "Status", "Error"]])

logs = pd.concat(logs, ignore_index=True)
assert len(logs) == 4300

# 2. Đối chiếu chính xác từng trạng thái với Evaluation
log_details = logs.rename(columns={"Status": "Log Status"})
details = ea.merge(log_details, on=["Run", "ID", "Method"],
                   how="left", validate="one_to_one")
assert len(details) == 4300
assert details["Log Status"].notna().all()
assert (details["Status"] == details["Log Status"]).all(), (
    "Status trong Evaluation và execution_log không nhất quán"
)

# 3. Phân loại lỗi theo thông điệp được lưu (phân loại gần đúng)
def classify_error(row):
    status = str(row["Status"]).upper()
    msg = str(row["Error"]).lower()

    if status == "TIMEOUT" or "timed out" in msg or "timeout" in msg:
        return "Timeout"
    if "syntax" in msg or "parser" in msg or "parsing" in msg or "parseexception" in msg:
        return "Syntax / Parser"
    if "binder" in msg or "catalog error" in msg or "column not found" in msg:
        return "SQL Schema / Binding"
    if "jdbc" in msg or "connection refused" in msg:
        return "Database Connection"
    if "unsupported" in msg or "not supported" in msg:
        return "Unsupported Operation"
    if any(x in msg for x in ["sqlgeneratorimpl", "questqueryprocessor", "reformulation"]):
        return "Ontop Stack Trace (unconfirmed cause)"
    return "Other / Unknown"

failures = details.loc[details["Outcome"] == "Execution Failure"].copy()
failures["Error Type"] = failures.apply(classify_error, axis=1)
assert len(failures) == int((ea["Status"] != "OK").sum())

print("=== EXECUTION FAILURE TAXONOMY ===")
taxonomy = pd.crosstab(failures["Method"], failures["Error Type"])
display(taxonomy)

print("\n=== SPARQL FAILURES BY QUADRANT ===")
display(pd.crosstab(
    failures.loc[failures["Method"] == "SPARQL", "Quadrant"],
    failures.loc[failures["Method"] == "SPARQL", "Error Type"]
))

print("\n=== HQHS FAILURE EXAMPLES ===")
display(
    failures.loc[
        (failures["Method"] == "SPARQL") & (failures["Quadrant"] == "HQHS"),
        ["Run", "ID", "Status", "Error Type", "Error"]
    ].head(10)
)

failures.to_csv(OUT_DIR / "execution_failure_details.csv", index=False, encoding="utf-8-sig")
taxonomy.to_csv(OUT_DIR / "execution_failure_taxonomy.csv", encoding="utf-8-sig")
print("Total execution failures:", len(failures))
print("Lưu ý: log có thể bị cắt ngắn, phân loại lỗi chỉ mang tính gợi ý.")

=== EXECUTION FAILURE TAXONOMY ===


Error Type,Ontop Stack Trace (unconfirmed cause),Other / Unknown,SQL Schema / Binding,Syntax / Parser
Method,,,,
SPARQL,249,0,0,33
SQL,0,1,80,0



=== SPARQL FAILURES BY QUADRANT ===


Error Type,Ontop Stack Trace (unconfirmed cause),Syntax / Parser
Quadrant,,
HQHS,185,20
HQLS,64,12
LQLS,0,1



=== HQHS FAILURE EXAMPLES ===


,Run,ID,Status,Error Type,Error
49,1,7,ERROR,Ontop Stack Trace (unconfirmed cause),hereSerializer$DefaultRelationVisitingSerializ...
61,1,19,ERROR,Ontop Stack Trace (unconfirmed cause),hereSerializer$DefaultRelationVisitingSerializ...
72,1,30,ERROR,Ontop Stack Trace (unconfirmed cause),hereSerializer$DefaultRelationVisitingSerializ...
80,1,38,ERROR,Ontop Stack Trace (unconfirmed cause),hereSerializer$DefaultRelationVisitingSerializ...
82,1,40,ERROR,Ontop Stack Trace (unconfirmed cause),hereSerializer$DefaultRelationVisitingSerializ...
158,2,30,ERROR,Ontop Stack Trace (unconfirmed cause),hereSerializer$DefaultRelationVisitingSerializ...
166,2,38,ERROR,Syntax / Parser,"uuid"" ...\n <IRI> ...\n ""if"" ...\n ""r..."
168,2,40,ERROR,Ontop Stack Trace (unconfirmed cause),hereSerializer$DefaultRelationVisitingSerializ...
169,2,41,ERROR,Ontop Stack Trace (unconfirmed cause),hereSerializer$DefaultRelationVisitingSerializ...
218,3,4,ERROR,Ontop Stack Trace (unconfirmed cause),hereSerializer$DefaultRelationVisitingSerializ...


Total execution failures: 363
Lưu ý: log có thể bị cắt ngắn, phân loại lỗi chỉ mang tính gợi ý.


In [5]:
# 1. Chọn các tình huống cần minh họa trong báo cáo
targets = [
    ("LQLS", "SPARQL Only"),
    ("HQLS", "SPARQL Only"),
    ("LQHS", "SPARQL Only"),
    ("HQHS", "SQL Only"),
    ("HQHS", "Both Wrong")
]

# 2. Đọc các truy vấn GPT-5.4 đã sinh, không gọi lại mô hình
generations = {}
for run_id in range(1, 51):
    file = (
        PROJECT_DIR / "results/llm/reproduction"
        / f"generation_run_{run_id:02d}.csv"
    )
    assert file.is_file(), file
    generation = pd.read_csv(file, keep_default_na=False)
    assert len(generation) == 86
    generation["ID"] = generation["ID"].astype(int)
    generation["Method"] = generation["Method"].astype(str).str.strip().str.upper()
    assert not generation.duplicated(["ID", "Method"]).any(), file
    generations[run_id] = generation

# 3. Mỗi loại chọn câu hỏi xuất hiện nhiều nhất, rồi chọn run nhỏ nhất
samples = []
for quadrant, case in targets:
    group = paired.loc[
        (paired["Quadrant"] == quadrant) & (paired["Case"] == case)
    ]
    if group.empty:
        print(f"Không tìm thấy trường hợp {quadrant} / {case}")
        continue

    qid = int(group["ID"].value_counts().idxmax())
    run_id = int(group.loc[group["ID"] == qid, "Run"].min())

    for method in ["SQL", "SPARQL"]:
        generation = generations[run_id]
        query = generation.loc[
            (generation["ID"] == qid) & (generation["Method"] == method)
        ].iloc[0]
        result = details.loc[
            (details["Run"] == run_id) &
            (details["ID"] == qid) &
            (details["Method"] == method)
        ].iloc[0]

        samples.append({
            "Quadrant": quadrant,
            "Case": case,
            "Run": run_id,
            "ID": qid,
            "Method": method,
            "Question": query["Question"],
            "Generated Query": query["Generated Query"],
            "Outcome": result["Outcome"],
            "Error": result["Error"]
        })

# 4. Xuất các ví dụ có đủ câu hỏi, truy vấn và thông báo lỗi
case_df = pd.DataFrame(samples)
assert not case_df.empty, "Không chọn được tình huống minh họa"
case_df.to_csv(OUT_DIR / "case_studies.csv", index=False, encoding="utf-8-sig")

print("=== CASE STUDIES ===")
display(case_df[["Quadrant", "Case", "Run", "ID", "Method", "Outcome"]])

unknown = failures.loc[failures["Error Type"] == "Other / Unknown"]
if not unknown.empty:
    print("\n=== UNKNOWN ERROR SAMPLE ===")
    print(unknown.iloc[0]["Error"])

print("Saved: case_studies.csv")

=== CASE STUDIES ===


,Quadrant,Case,Run,ID,Method,Outcome
0,LQLS,SPARQL Only,1,17,SQL,Wrong Result
1,LQLS,SPARQL Only,1,17,SPARQL,Correct
2,HQLS,SPARQL Only,1,28,SQL,Wrong Result
3,HQLS,SPARQL Only,1,28,SPARQL,Correct
4,LQHS,SPARQL Only,1,5,SQL,Wrong Result
5,LQHS,SPARQL Only,1,5,SPARQL,Correct
6,HQHS,SQL Only,1,38,SQL,Correct
7,HQHS,SQL Only,1,38,SPARQL,Execution Failure
8,HQHS,Both Wrong,1,1,SQL,Wrong Result
9,HQHS,Both Wrong,1,1,SPARQL,Wrong Result



=== UNKNOWN ERROR SAMPLE ===
Not a SELECT/WITH query
Saved: case_studies.csv
